# Day 12 — NLP & Classical Text Processing

RetailMind AI review analytics: NLTK cleaning and tokenization, TF-IDF (unigrams and bigrams), logistic regression sentiment classification, holdout evaluation, informative terms and reusable inference.

**Dataset:** Amazon Review Polarity, original CSV version ([Kaggle](https://www.kaggle.com/datasets/kritanjalijain/amazon-reviews), [fast.ai mirror](https://course20.fast.ai/datasets), [Hugging Face equivalent](https://huggingface.co/datasets/fancyzhx/amazon_polarity)). Original CSV labels: 1 = negative, 2 = positive; mapped to 0 and 1. The CSVs are headerless and have three columns: label, title, content.

Use the original train/test split and sample up to 10,000 training reviews and 2,000 test reviews per class. No earlier notebook is required. The full dataset is not loaded into RAM. No embeddings or BERT are used in this classical baseline.

## 1. Setup

Install NLTK in the existing project environment (`pip install nltk`). The tokenizer does not require NLTK corpus downloads. Stopwords are from scikit-learn, with negation words retained. Run from the repository root or the `notebooks/` directory.

In [ ]:
from pathlib import Path
import sys
import json
import csv
import shutil
import tarfile
from urllib.request import urlretrieve
from collections import Counter

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

ROOT = Path.cwd().resolve()
if not (ROOT / 'src' / 'review_sentiment.py').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'review_sentiment.py').is_file():
    raise FileNotFoundError('Place review_sentiment.py inside the repository src/ folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.review_sentiment import (
    clean_review, load_amazon_reviews, build_sentiment_pipeline,
    informative_terms, save_sentiment_model, predict_reviews
)

DATA_DIR = ROOT / 'data' / 'amazon_polarity'
MODEL_PATH = ROOT / 'models' / 'review_sentiment_day12.joblib'
METRICS_PATH = ROOT / 'models' / 'review_sentiment_day12_metrics.json'
TERMS_PATH = ROOT / 'docs' / 'day12_sentiment_terms.csv'
SEED = 42
TRAIN_PER_CLASS = 10000
TEST_PER_CLASS = 2000
AUTO_DOWNLOAD = True

## 2. Load Amazon reviews

If `data/amazon_polarity/train.csv` and `test.csv` already exist, they are reused. Otherwise this cell downloads the original archive (roughly 690 MB compressed) from the fast.ai public mirror using Python's standard library. To download manually, use the Kaggle link above, extract the two CSVs into this folder and set `AUTO_DOWNLOAD = False`. The extracted files require additional disk space. Only the CSVs are extracted.

Reservoir sampling selects a reproducible random subset of each class from the entire official split. The test set is not used for fitting.

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
train_path = DATA_DIR / 'train.csv'
test_path = DATA_DIR / 'test.csv'

if not (train_path.is_file() and test_path.is_file()):
    archive_path = DATA_DIR / 'amazon_review_polarity_csv.tgz'
    if not archive_path.is_file():
        if not AUTO_DOWNLOAD:
            raise FileNotFoundError(f'Place train.csv and test.csv in {DATA_DIR}')
        url = 'https://s3.amazonaws.com/fast-ai-nlp/amazon_review_polarity_csv.tgz'
        partial = archive_path.with_suffix('.part')
        print('Downloading Amazon Polarity archive...')
        urlretrieve(url, partial)
        partial.replace(archive_path)

    print('Extracting train.csv and test.csv...')
    with tarfile.open(archive_path, 'r|gz') as archive:
        for member in archive:
            name = Path(member.name).name
            if member.isfile() and name in {'train.csv', 'test.csv'}:
                target = DATA_DIR / name
                if not target.exists():
                    partial = target.with_suffix('.part')
                    with archive.extractfile(member) as source, partial.open('wb') as dest:
                        shutil.copyfileobj(source, dest)
                    partial.replace(target)
            if train_path.is_file() and test_path.is_file():
                break
    if not (train_path.is_file() and test_path.is_file()):
        raise FileNotFoundError('The archive did not contain both required CSV files.')

X_train, y_train = load_amazon_reviews(train_path, TRAIN_PER_CLASS, SEED)
X_test, y_test = load_amazon_reviews(test_path, TEST_PER_CLASS, SEED + 1)

# Exclude exact raw-text overlap with the sampled training set.
train_texts = {text.casefold().strip() for text in X_train}
heldout = [(text, label) for text, label in zip(X_test, y_test)
           if text.casefold().strip() not in train_texts]
removed = len(X_test) - len(heldout)
if not heldout or len({label for _, label in heldout}) != 2:
    raise ValueError('Both classes must remain in the test set after overlap removal.')
X_test, y_test = map(list, zip(*heldout))

print('Train:', len(X_train), Counter(y_train))
print('Test:', len(X_test), Counter(y_test))
print('Exact train/test overlap removed:', removed)
print('Sample review:', X_train[0][:300])

## 3. Text cleaning

Convert to lowercase, decode/remove HTML, remove URLs, expand negative contractions, tokenize with NLTK, remove non-letter noise and stopwords. Keep negation terms such as `not`, `no` and `never`. No stemming or lemmatization is required. The same tokenizer will be used at training and inference time.

In [ ]:
for review in [X_train[0], "<b>I didn't like it!</b> Visit https://example.com. Not worth the money."]:
    print('Original:', review[:300])
    print('Cleaned: ', clean_review(review)[:300])
    print()

## 4. TF-IDF and classical sentiment baseline

TF-IDF gives higher weight to discriminative words and two-word phrases. Logistic regression learns positive and negative sentiment from the sparse vectors. The pipeline fits its vocabulary and IDF using only training data. The test set is transformed with the fitted vocabulary.

In [ ]:
model = build_sentiment_pipeline(max_features=30000, min_df=2)
model.fit(X_train, y_train)

vectorizer = model.named_steps['tfidf']
print('Vocabulary size:', len(vectorizer.get_feature_names_out()))
print('Training TF-IDF shape:', (len(X_train), len(vectorizer.get_feature_names_out())))
print('Example transformed shape:', vectorizer.transform(X_train[:5]).shape)

## 5. Evaluate the model

Accuracy, positive-class precision/recall/F1, macro F1 and ROC-AUC are measured on the separate official test subset. The majority-class baseline provides a simple comparison. The confusion matrix uses actual labels as rows and predicted labels as columns (negative, positive).

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, list(model.classes_).index(1)]
metrics = {
    'accuracy': float(accuracy_score(y_test, y_pred)),
    'precision': float(precision_score(y_test, y_pred, zero_division=0)),
    'recall': float(recall_score(y_test, y_pred, zero_division=0)),
    'f1': float(f1_score(y_test, y_pred, zero_division=0)),
    'macro_f1': float(f1_score(y_test, y_pred, average='macro', zero_division=0)),
    'roc_auc': float(roc_auc_score(y_test, y_prob)),
    'majority_baseline_accuracy': max(Counter(y_test).values()) / len(y_test),
}
for name, value in metrics.items():
    print(f'{name:28s}: {value:.4f}')

print('\nClassification report:')
print(classification_report(y_test, y_pred, labels=[0, 1],
                            target_names=['Negative', 'Positive'], zero_division=0))
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print('Confusion matrix (negative, positive):')
print(cm)

## 6. Most informative sentiment terms

Logistic regression assigns a signed coefficient to each TF-IDF feature. Large negative coefficients are associated with negative sentiment, and large positive coefficients with positive sentiment. These are model associations, not verified product defects or causal explanations. Review text should be inspected before turning a term into a business action.

In [ ]:
terms = informative_terms(model, top_n=15)
for sentiment, entries in terms.items():
    print(f'\n{sentiment} sentiment terms')
    for term, weight in entries:
        print(f'{term:25s} {weight:+.4f}')

## 7. Save model and results

Persist one joblib pipeline containing the tokenizer reference, fitted TF-IDF vocabulary and classifier. Keep `src/review_sentiment.py` with the model, since joblib needs its tokenizer at load time. Use matching library versions and load only trusted artifacts. Metrics and top terms are generated from this run.

In [ ]:
save_sentiment_model(model, MODEL_PATH)
summary = {
    'dataset': 'Amazon Review Polarity (original CSV)',
    'train_rows': len(X_train), 'test_rows': len(X_test),
    'train_class_counts': dict(Counter(y_train)),
    'test_class_counts': dict(Counter(y_test)),
    'removed_exact_overlap': removed,
    'seed': SEED, 'vocabulary_size': len(vectorizer.get_feature_names_out()),
    'metrics': metrics, 'confusion_matrix': cm.tolist(),
}
METRICS_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')
TERMS_PATH.parent.mkdir(parents=True, exist_ok=True)
with TERMS_PATH.open('w', encoding='utf-8', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['sentiment', 'term', 'coefficient'])
    for sentiment, entries in terms.items():
        writer.writerows((sentiment, term, weight) for term, weight in entries)

print('Model:', MODEL_PATH)
print('Metrics:', METRICS_PATH)
print('Terms:', TERMS_PATH)

## 8. Predict new product reviews

Exercise the saved pipeline on raw text, with no manual preprocessing needed. The positive probability is the logistic regression estimate, not a guaranteed or independently calibrated confidence score.

In [ ]:
reviews = [
    'Excellent quality and fast delivery. Highly recommended!',
    'The product stopped working after two days. Very disappointed.',
    'Not worth the money. I want a refund.',
    'The packaging was damaged but the product works perfectly.'
]
for result in predict_reviews(reviews, MODEL_PATH):
    print(result['review'])
    print(f"Sentiment: {result['sentiment']} | P(positive): {result['positive_probability']:.4f}\n")

# Verify that reloading the model preserves predictions.
reloaded = predict_reviews(reviews, MODEL_PATH)
expected = model.predict_proba(reviews)[:, list(model.classes_).index(1)]
assert all(abs(row['positive_probability'] - prob) < 1e-10
           for row, prob in zip(reloaded, expected))
print('Saved pipeline round-trip verified.')

## Conclusion

Day 12 adds reusable binary review sentiment analytics to RetailMind AI. The recorded holdout metrics and learned terms are written above; they are the baseline for the later BERT increment. Limitations: no neutral class, no product identifiers in this dataset, and limited handling of sarcasm, mixed sentiment and context. The model does not directly perform product-level issue extraction or support-ticket routing.